# SCALING LAWS FOR SPARSELY-CONNECTED FOUNDATION MODELS

### 1 Введение

Успех трансформерных моделей, как текстовых так и визуальных, обоснован в том числе их предсказуемым масштабированием, что было отражено в предложенных Kaplan в 2020 году законах масштабирования

Параллельно идут попытки ускорить вычисления с помощью квантизации или прунинга. Однако, несмотря на очевидный интерес, возможности масштабирования в разреженных моделях до сих пор нормально не были изучены.

В этой работе мы исследуем, как введение разреженности влияет на масштабирования больших языковых моделей. Конкретно, мы фокусируемся на пруниге (уровня отдельных весов) в трансформерах, текстовых так и визуальных.

Мы используем большие датасеты JFT-4B (изображения) и C4 (текст), которые на несколько порядков больше тех, на которые обычно ссылаются в работах, исследующих разреженность. На таких объемах модели масштабируются условно бесконечно, поэтому вообще не ясно, смогут ли разреженные модели обогнать плотные при одинаковой квоте на compute. На старых бенчмарках из-за ограниченных объемов выборок модели упирались в потолок быстро, поэтому разреженные модели часто выглядели предпочитетльнее - они обучались быстрее (при том же compute) и без проблем догоняли плотные модели.

Чтобы оценить влияние разреженности, мы предлагаем свой обобщенный закон масштабирования (scaling law), учитывающий помимо N и D ещё разреженность S и показываем, что он хорошо работает как для текста так и для изображений:

$$
L(S, N, D) = \left( a_S (1 - S)^{b_S} + c_S \right) \cdot \left( \frac{1}{N} \right)^{b_N} + \left( \frac{a_D}{D} \right)^{b_D} + c
$$

Первое слагаемое отражает "емкость" модели (общее кол-во параметров, скорректированное на степень разреженности), второе слагаемое отражает зависимость от длины обучения D, третье - неуменьшаемую ошибку. 

Сама же степень разреженности $1-S$ моделируется отдельной показательной функцией. Как в Каплане, экспоненты $b_S$ и $b_N$ нужны для подгонки конкретного вида зависимости

Мы проверили корректность формулы эмпирически на сетке (N,D,S)

<img src="img/pruning1.png" width=500>


Можем сделать следующие важные выводы:
- Разреженность модели корректирует влияние N мультипликативно, как некий коэффициент.  При этом влияние времени обучения D почти никак не корректирует от разреженности
- Можем определить оптимальный режим S для заданного бюджета на вычисления, при котором разреженность выгоднее, чем просто дообучить плотную модель
- Графики с оптимальной разреженностью параллельны графикам плотных моделей Chinchilla. Более того, чем больше бюджет, тем выше можно ставить разреженность. В то время как в “плотном мире” множество оптимальных (N,C) моделей образует прямую, в разреженном множество оптимальных моделей образует целую полуплоскость (S+N,C)
- Мы также изучили hardware friendly m:n прунинг. Пост-прунинг на чекпойнтах работает лучше, чем динамический [уточнить]

Итого, мы предоставили первый закон учитывающий прунинг. С точки зрения теории он позволяет лучше понять сильные и слабые стороны  разреживания для задач/моделей. С точки зрения практики - оценить её перспективность для ускорения инференса/обучения, когда есть сотовыествующие инструменты

### 2 FAIR EVALUATION IN THE PRESENCE OF STRONG SCALING

Предыдущие бенчмарки требуют пересмотра, так как подходили только малым моделям. Трансформеры же по-другому масштабируются. 
Почему:
- Data: На старых pruning бенчмарках (где используются небольшие датасеты) вместо параллельного обучения 2х сравниваемых моделей, делали так: сначала обучают Dense модель, потом её прунят, дооубучают примерно на стольких же токенах и сравнивают было/стало. Так делали потому что обучать было сложнее + важно был именно финальный результат. Там это норм, поскольку модель быстро насыщается и обучение прунингу. В случае с большими моделями, они дообучаются условно всегда, поэтому за время повторного обучения performance скорее всего вырастет

- Model size.   Developing small but accurate dense models used to require arranging many custom modules into a carefully engineered - architecture (Howard et al., 2017; Tan & Le, 2019). Naively scaling down a 25M parameter ResNet50 by a factor of 10 will not yield a competitive 2.5M parameter ImageNet model, which is why most pruning papers omit a comparison against such a variant. However, when considering Transformer models and massive datasets, basic width and depth scaling typically results in a very strong family of differently-sized models. Hence, it is critical to always compare sparse models with a dense version of equivalent parameter count. 

- Computational costs. Вычислительная оптимальность по Каплану обычно не учитывается в старых бенчмарках - считается, что там все модели и так обучаются до сходимости, поскольку обучение там дешевое. Однако небольшой трансофрмер, обученый достаточно долго может обогнать по качеству трансформер обученый на том же бюджете (с меньшем кол-ве шагов). Это подчеркивает сложность честного сравнения. Например, разреженную на 50% предобученную на 100K модель было бы честно сравнивать с N/2 плотной моделью, обученой на том же бюджете, то есть на 200K шагов (еще и с небольшой поправкой на сам прунинг).

Резюмируя, в действительно “честном” процессе прунинга разреженность не должна использовать (leverage) продленное обучение или специальные оптимизации архитектуры. Иначе сравнения будут всегда в пользу разреженных моделей, плотные модели не смогут использовать свои преимущества. Мы не уверены, что разреженные трансформеры в принципе могут побеждать, если мы сравниваем честно с учетом всех факторов. Для ответа нужна общая формула.

## 3 SCALING LAWS FOR PARAMETER-SPARSE TRANSFORMERS 

#### 3.1 Дизайн экспермиента
В качестве тестируемых больших моделей взяли ViT, обученый под классификацию на JFT и T5, обученый под MLM на C4. Разреживание (gradual magnitude pruning) делали в процессе обучения между 25% и 75% точками.

Сетка для визуальных моделей: 2x по размерам, 4 варианта датасетов, компрессия: 2,4,8<br>Сетка для текстовых: 4x, 3 варианта обучения, компресси: 2,4,8. В целом, нам устойчивость процесса была важнее эффективности прунинга

#### 3.2 Как выводилась формула
По Каплану валидационный лосс считается по формуле

$$
L(N, D) = \left( \frac{a_N}{N} \right)^{b_N} + \left( \frac{a_D}{D} \right)^{b_D} + c
$$

Первые два млагаемых - про N и D, третье - неуменьшаемый шум. Степени довольно стабильны, а мультипликаторы могут немного меняться при изменении архитектуры или алгоритма обучения.

Как и там, мы предполагаем, что данные и compute бесконечны (не повторяются). Мы специально рассматриваем именно задачу pre-training и бесконечные данные, так как она задает максмиальные требования к разреженности. На более узких задачах она может использоваться меньше. 

Главный вопрос - как именно S входит в закон масшатибрования. Тогда сможем определить и оптимальную степень рареженности и предельный performance. Пока толькло вопросы: большие модели легче разрежать? разреженные модели получают больше выгоды от более длительной тренировки? остальные параметры не зависят от рареженности? 

Чтобы получить какие-то инсайты запустили sweep search для модели T5. По результатам построили картинку (см ниже). Везде шкалы log-log и вычли 1

<img src="img/pruning_results.png" width=750>

Три главных наблюдения:
- разные S дают почти параллельные линии
- разреженные модели точнее, но быстрее заканчивают обучаться
- разные D также дают почти параллельные линии

Они нам диктуют следующее: S влияет на N как мультипликатор, наклон (показатель функции) остается таким же. При этом S моделируем, как отдельный степенной закон, его влияение перестает действовать с какого-то момента. А S вообще не влияет на D, его оставляем как в оригинальной формуле

$$
L(S, N, D) = 
\left( 
\textcolor{blue}{a_S (1 - S)^{b_S} + c_S} 
\right) \cdot 
\left( 
\textcolor{teal}{\frac{1}{N}} 
\right)^{\textcolor{teal}{b_N}} + 
\left( 
\textcolor{brown}{\frac{a_D}{D}} 
\right)^{\textcolor{brown}{b_D}} + 
\textcolor{black}{c}
$$

Чтобы отразить факт, что 75% разреженность в два раза “разреженнее” 50%-ой, мы включаем этот показаетль, как compression rate 1/(1 − S). В итоге формула состоит из capacity (size и sparsity) и data
Формула говорит нам, что высокая разреженность всегда лучше. Однако с какого-то момента (64x) начинает откатываться назад. Возможно, эту зависимость было бы лучше моделировать квадратом, но мы считаем, что это все-таки граничные случаи. Ну и когда S = 0, наш закон повторяет оригинальный


Подгоняем коэффициенты формулы, минимизируя Huber loss от LogL BFGS с δ = 0.001 (чтобы был чуть робастнее) с несколькими рандомными инциализациями. Для проверки построили график Actual vs Prediction - визуально не придирешься

Далее мы решили запрунить большую 2.3B модель на 75% (это примерно в an ≈ 6.75× больше максимальной модели в нашей обучающей выборке). Примерно такой же уровень экстраполяции был в исследованиях Шиншилы. Для более эффективного обучения [и чтобы избежать bottlenecks], мы решили взять T5-XL версию модели и обучали ее на 250K батчей по 256 наблюдений в каждом (вместо 500 батчей по 128). Несмотря на эти изменения, предсказанный валидационный лосс довольно точно совпал с наблюдаемым. 

Наконец, мы сделали аналогичный grid search для ViT модели и повторили процесс подгонки. Тут выбрали параметры посвободнее δ = 0.01, так как . При этом наблюдений чуть больше, коэффициенты предсказываются точнее. Сравнили предсказанные значения с наблюдаемыми - все совпадает, только на совсем малых D точки отходят от закона.

#### 3.3 Оптимальная разреженность
Имея общий scaling law, мы теперь можем сравнить как показывают себя модели разной степени разреженности при одинаковом кол-ве активных параметров (N) и бюджете на compute (C).
Оптимальная степень разреженности Sopt(N, C) дает наименьший validation loss для заданных N и C.

Стоимость обучения полной модели считается как $C = 6ND$ (почему - см Kaplan). Поскольку в нашем случае от N только часть ненулевых весов, выбирать бюджет дольше, значит считаем по скорректированной формуле $C = 6ND \cdot c_{mul}$. 

При этом сам коэффициент $c_{mul}$ можно посчитать двумя способами: 
1. без учета плавности обучения<br>
   $c_{mul}(S) = \frac{1}{1 − S}$
2. с учетом плавности обучения [25%-75%]<br>
   $c_{mul}(S) = \frac{(0.25+0.50·(1−0.75·S))}{(1−S)}+0.25$

Берем основную формулу<br>
$
L(N, D, S) = a_S (1 - S)^{b_S} N^{-b_N} + a_D D^{-b_D}
$

Выражаем $D$ через N<br>
$
C = c_{\text{mul}}(S) \cdot N \cdot D
\quad \Rightarrow \quad
D = \frac{C}{c_{\text{mul}}(S) \cdot N}
$

Подставляем в общую формулу<br>
$
L(S) = a_S (1 - S)^{b_S} N^{-b_N} + a_D C^{-b_D} \cdot c_{\text{mul}}(S)^{b_D} \cdot N^{b_D}
$

Считаем производную по $S$<br>
$
\frac{dL}{dS} = -a_S b_S (1 - S)^{b_S - 1} N^{-b_N} + a_D C^{-b_D} N^{b_D} \cdot \frac{d}{dS} \left[ c_{\text{mul}}(S)^{b_D} \right]
$

Приравниваем эту производную нулю<br>
$
a_D C^{-b_D} N^{b_D} \cdot \frac{d}{dS} \left[ c_{\text{mul}}(S)^{b_D} \right]
= a_S b_S (1 - S)^{b_S - 1} N^{-b_N}
$

Или так<br>
$
a_D \cdot \frac{d}{dS} \left[ c_{\text{mul}}(S)^{b_D} \right]
= a_S b_S C^{b_D} (1 - S)^{b_S - 1} N^{-(b_N + b_D)}
$

Данное место точек $(C,N)$ задает линию оптимальности где точность модели разреженности $S$ наибольшая ($S$ оптимальна):
$$ a_D \cdot D^{\frac{c_{\text{mul}}(S)}{c_{\text{mul}}(S)}} \cdot \left( \frac{D_S}{c_{\text{mul}}(S)} \right)^{-b_D}
= a_S b_S (1 - S)^{b_S - 1} \cdot N^{-b_N}$$

An interesting property about this contour is that it implies DS = O(N bN /bD ), meaning that if data- is stronger than size-scaling, then the same sparsity is optimal for a smaller data-to-size ratio on larger models. This is sensible as a process bottlenecked more by capacity than by data will benefit more from increasing the former, e.g., by adding sparsity. 

Для костов посчитаных без учета плавности обучения $c_{mul}(S) = \frac{1}{1 − S}$, можно просто решить уравнение одной перемнной и получить: 

$$
S_{\text{opt}}(N, C) = \max \left\{ 
1 - \exp \left( 
\frac{
\log\left( \frac{b_N a_D b_D}{a_S b_S} \right) + b_N \log N
}{
b_D + b_S
} 
\right)
\cdot 
\left( \frac{C}{6N} \right)^{- \frac{b_D}{b_D + b_S}},\ 0 
\right\}
$$

Empirical results. Посчитаем кривые оптимальности для моделей T5 и ViT, которые мы исследовали в предыдущем параграфе. 
Figure 1 (Right) and 4 show the optimal sparsity contours, both for dense and sparse costs. 
И по построению формулы, и по графикам видно, что линии оптимальности параллельны линиям оптимальности по Шиншилле (которые показывают идеальное разложение C на N и D для полных моделей).  However, we note that the Chinchilla line does not necessarily correspond to the S = 0 case since non-zero sparsity may be optimal in this regime (this is the case for sparse-FLOPs).      

Гланый вывод - The key take-away from these results is that as one trains significantly longer than Chinchilla (dense compute optimal), more and more sparse models start to become optimal in terms of loss for the same number of non-zero parameters. This is because the gains of further training dense models start to slow down significantly at some point, allowing sparse models to overtake them. We further illustrate this effect on a subset of our actual ViT data in Figure 5. 
The practical question now is how much longer training is necessary? In terms of sparse FLOPs, 50% sparsity is already optimal for < 2× (ViT) and < 3× (T5) longer training than Chinchilla; for dense FLOPs it is ≈ 5× and ≈ 70×, respectively. While the latter number seems quite high at first glance, we note that language models of the sizes we consider here are already typically trained for > 100× longer than Chinchilla (Brown et al., 2020). Additionally, larger models are being trained with more and more data as well, e.g., Llama2-7B with ≈ 14× Chinchilla (Touvron et al., 2023b). In general, the optimal sparsity at a given point (N, C) is lower for dense than sparse FLOPs since the former assumes that sparsity provides no benefits during training. 
Figure 5: Loss vs. sparse pretraining FLOPs for ViT models of varying sparsity. 

#### 3.3.1 Эффективность прунинга
Мы выяснили, при каком соотношении N,D разреженные до S модели дают наилучшее качество. Но пока не знаем, насколько разреживание может сжимать полную модель без потери качества. При этом компьюта потребуется дольше, несмотря 

Вопрос - насколько больше плотная модель должна быть больше разреженной, чтобы обе после обучени давали одинаковую точность? Ответ обзовем метрикой Gain 

Почему называется Gain? Потому что насколько больше плотная = насколько меньше рареженная - а это и есть наша цель, сэкономить веса. Что-то типа коэффициента компрессии. Если он большой, значит мы эффективно пруним.

Берем разреженную S модель с $N$ ненулевыми весами. Ищем более круную плотную модель с $G \cdot  N$ ненулевыми весами так, чтобы лосс у них был эквивалентным:
$$
\alpha_S (G \cdot N)^{-b_N} + \alpha_D D^{-b_D} + L_\infty = \left[a_S (1 - S)^{b_S} + c_S \right] N^{-b_N} + \alpha_D D^{-b_D} + L_\infty
$$

Сокращаем все, что одинаковое:
$$
G^{-b_N} = \frac{a_S (1 - S)^{b_S} + c_S}{a_S + c_S}
$$

Итого, метрика Gain равна:
$$
\text{gain}(S) = \left( \frac{a_S (1 - S)^{b_S} + c_S}{a_S + c_S} \right)^{-1 / b_N}
$$

Посчитанные значения метрики приведены в таблице ниже. Для 75% разреженной модели ViT соответсбщая по качеству полная модель должна быть размером 2.17x от кол-ва ненулевых параметров  

Важно что от размера этот эффект не зависит. Crucially, this holds for any amount of data and thus also in the infinite limit when training is purely capacity bound. Hence, this expresses an equivalence between dense capacity and sparse capacity. 

Что характерно, и для текста и для изображений выигрыши почти одинаковые с наилучшими значениями в районе S=75% 

<img src="img/pruning_perf_limit.png" width=300>

## 4 Дополнительно 

#### 4.1 Разреженности вида N:M
Рассмотрим также "структурную разреженность" вида n:m (не более n весов по строке и m по столбцу), которая хорошо оптимизируется на уровне железа

<img src="img/pruning_types.png" width=500>

Известно, что мелкие алгоритмические изменения влияют только на масштабирующие коэффициенты, поэтому здесь также ожидаем влияние только на sparsity слагаемое. Значит, Если график для плотной модели уже построен, то нам достаточно подогнать только $a_S, b_S, c_S$, чтобы пересчитать L(S, N, D). Поэтому мы не делаем полный пересчет, а обучаем только наименьшую модель или наименьшее кол-во шагов

<img src="img/pruning_figure_6.png" width=300>

На графике 6 приведены полученные результаты и они очень похожи на результаты графика 2: форма закономерности совпадает с общим законом. Подгоняем формулу с коэффициентом Huber δ = 0.01 и 0.75 и считаем экономию от спрасификации (see Table 3). В целом видно, что разреживание 2:4 и 4:8 дает прирост похожий на 50% (see Table 2 and also Fig- ure 6), хотя n:m вероятно более зашумленеы в виду меньшего кол-ва доступного точек. Между тем 1:4 почти не приносит экономии, а 2:8 приносит чуть-чуть, что немного ломает результат. Мы предполагаем, что 75% [too stringent to significantly increase capacity beyond their 50% variants] 

#### 4.2 Прунинг предобученых моделей
Наконец рассмотрим кейс, когда мы хотим оптимизировать набор обученных моделей. Может потратиться на pre-training. Сравним 2 режима прунинг с нуля прунинг из предобученой модели. Мы обучаем 3 визуальные модели S/16, M/16 и B/16 на датасете JFT в течение 4 эпох, а затем запускаем прунинг на 5.6% бюджета (не ждем 25% обучения. Далее испольщуем формулу посчитать сколько нужно обучения для модели аналогичной точности. В таблице выведено насколько менбше больше данных нужно чтобы обучить запруненуб моджель с нуля

Table 4: сколько нужно данных для прунинга модели с нуля, чтобы валидационный лосс был таким же, как после прунинга предобученой модели. exc = исключая стоимость самого предобучения, inc = включая 

Если модель уже предобучена, начинать прунить с чекпоинта, то это в более чем в 4 раза эффективее, чем обучать 50%/75% модели с нуля.  If the model already exists and there is thus no pretraining cost, then starting from such a checkpoint is > 4× more efficient then sparsifying from scratch for 0.5/0.75, and > 2× for 0.875 sparsity, respectively. The reason why the efficiency gains are decreasing with higher sparsity is most likely the increased divergence from the initial starting point. At the same time, when the pretraining cost is counted as well, pruning throughout the whole training process appears to be ≥ 4× more efficient, relative to the ≈ 5% pruning of pretraining budget. 

Результаты показывают что хотя прунинг выигрывает от наличия предобученой плотной модели, это работает до определенного увроня. И 50% модели  Finally, we note that the 50% models are ≈ 0.2 − 0.3 points away from their dense baseline loss, which matches our results in Section 3.3.1 that the size gain of 50% sparsity is noticeably less than 2× for well trained models. 

## 5 Другие работы
Исследование разреженности и прунинга сетей имеют долгую историю, которая начинается ещё с работ Яна Лекуна. Среди текущих SOTA решений можно выделить:
- gradual removal 
- разреженное обучение 
- Hessian based 
- soft оптимищация

Эти методы позволяют занулить много весов с минимальными просадкой по точности, что дает прирост по скорости, если используются специальные inference алгоритмы. Однако, большинство из них фокусируются на относительно простых задачах, при этом проверяются многопараметрические модели.

В то же время мало работ про прунинг трансофрмеров. Например, Gopher экспериментировали с прунингом и увидели, что разреженные модели могут обгонять полные модели при одинаковом кол-ве обучения. Но оставили открытым вопрос, [whether this is also possible when accounting for the significantly increased compute spent for producing those sparse models, relative to dense ones trained with the same amount of data/steps] 
Также Cerebras прунили GPT модель, используя сущесвтенно больше обучения, чем для своей полной модели. Недавно SparseGPT показали что можно прунить большие модели даже без их переобучения но не понятно, можно ли так делать [on more recent, smaller and much less undertrained networks]

Огромный успех трансформеров обязан их возможностям по масшатибррованию: увеличив N и/или D, гарантировано получаем прирост качества, даже если модель уже большая. Прирост хорошо прогнозируем и моделируется простым степеными законом. This can, for example, be utilized to construct a family of training compute optimal models (Hoffmann et al., 2022). В последнее время этот закон экстраполировали на много более узких доменов, например, выбор оптимальной архитектуры, MoE, multi-epoch обучение на одних и тех же данных, и некоторые прикладыне задачи. Однако мало ислледовалось влияние разреженности.

Rosenfeld et al. (2021) studies the relationship between width, depth and weight-density for pruning pretrained ResNets trained primarily on the nowadays very small CIFAR10 dataset. Contrarily, we consider modern Transformers trained on datasets many orders of magnitude larger and focus particularly on the data/compute dimension that is crucial in this context, but not very relevant in the setting of Rosenfeld et al. (2021). 

Transformer efficiency. Overall, making (large) Transformers more efficient is currently a highly active area of research. Probably the currently most popular and practical approach is quantization, that is reducing the numerical precision of weights (and sometimes also activations) (Frantar et al., 2022; Dettmers & Zettlemoyer, 2022; Xiao et al., 2022). Further, there are also many works on Mixture-of-Expert (MoE) models, large ensembles of models/individual layers where each input is only processed by a small part, thus keeping the overall computation cost constant (Du et al., 2022; Fedus et al., 2022; Artetxe et al., 2022; Riquelme et al., 2021). MoEs are a form of dynamic activation sparsity, which is very different from the static weight sparsity that we study in this work; the former trades off increased memory for faster inference, whereas the latter reduces both inference and memory costs. In general, we note that quantization, MoEs and weight sparsity are all complementary techniques that may be stacked for compound gains (Han et al., 2016; Kurtic et al., 2022). 

## 6 Обсуждение
Несмотря на наши многочисленные эксперименты, результаты имеют некоторые ограничения, которые мы надеемся устранить в будущих работах 
- Мы сделали фокус на устойчивость и масштабируемость между разными постановками, не для максимизации эффективности под конкретные задачи. Хотя мы уверены, что вид результата останется тем же, коэффициенты можно настоить лучше, более широким перебором или другими стратегиями пурнинга
- В нашей работе мы делали прунинг под задачу предобучения. [С точки зрения практики, это удобно, приложения выигрывают от работы с более эффективной моделью, это усложняет компрессию.] Мы думаем, что разреженности могут быть существенно улучшены, если прунинг делается под конкретные приложения, для которых важны не все, а только некоторые возможности такой модели. Также мы подразумевали наличие бесконечного набора данных, что исключает переобучение from dense baselines. Мы думаем, что разреженность может быть особенно полезна когда данные ограничены и многократно переиспользоваться
- Наша цель была понять общие закономерности, поэтому мы ориентировались на самую простую метрику: кол-во ненулевых весов. На практике ускорение разреженностью работает сложнее, железки часто не дают идеальную масштабируемость и есть операции типа attention и нормализации, которые не выигрывают от разреживания. Мы планируем рассмотреть и другие метрики тоже в будущих работах

Далее рассмотрим как наши выводы бьются с резльуттатми других исследованмий
- Rae et al. (2021) изучали GPT подобные модели генерации, их результаты совпадают с нашими, их выигрыш в 2.5x для 90% разреженности близки к нашим выводам
- С другой стороны Cerebras (2022) репортуют намного лучший результат 5x, но в другой постановке где бейзлайн оптимальный по обучению (не инфересу) и прунинг использщует 5x больше данных чем dense comparison point. Это не бьется с нашими результатами: по нашему закону, подогнанному для T5 модели, должно ьбыть 1.5 и 1.48
- Finally, SparseGPT (Frantar & Alistarh, 2023) notes that post-training pruning becomes signif- icantly easier as the model size increases. However, they do not perform any retraining, and observe this effect primarily relative to the respective unpruned base model, not in terms of im- provements over the Pareto size-vs-loss frontier that we study in this work. Hence, we believe that this is likely more related to the pretrained models’ initial robustness to pertubations rather than the architecture’s inherent sparsifiability. 
Practical consequences. Our scaling insights lead to a number of practical consequences: Spar- sity seems to affect each model size in approximately the same way, while remaining mostly inde- pendent of the amount of training data used. This provides evidence that good pruning performance in less expensive settings should generalize to performance at scale, which will hopefully accelerate research on new sparsification recipes and algorithms. Additionally, we have shown that optimal sparsity levels continuously increase with longer training. Sparsity thus provides a means to further improve model performance for a fixed final parameter cost. In particular, when training beyond Chinchilla optimality, where simple dense training starts to run into diminishing returns, sparsity can provide a clear alternative. Thus, our findings can be interpreted as providing practical motiva- tion for further developing sparsity support. 


In [ ]:
$$cmul(S) = (0.25+0.50·(1−0.75·S))/(1−S)+0.25$$